# Train/Validation/Test Split + Feature Scaling

**Week 4 - Task B**  
**Author:** Pratik (Team Lead)  
**Date:** February 2026

## Objectives
1. Load engineered features from Yugant's work (Task A)
2. Create stratified train/validation/test splits (70/15/15)
3. Apply StandardScaler to prevent data leakage
4. Save scaler object for future use
5. Verify balance across splits and check for data leakage
6. Save all splits for Week 5 modeling

---

## 1. Setup & Imports

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os
import warnings

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Libraries imported successfully!")
print(f"✓ Random seed set to: {RANDOM_STATE}")

---
## 2. Load Engineered Features

Load the dataset created by Yugant in Task A (feature engineering + one-hot encoding).

In [ ]:
# Load engineered features
df = pd.read_csv('../data/processed/engineered_features.csv')

print("=" * 70)
print("ENGINEERED FEATURES LOADED")
print("=" * 70)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum().sum()
print(f"\nMissing values: {missing_values}")

if missing_values > 0:
    print("\n⚠️  WARNING: Missing values detected!")
    print(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print("✓ No missing values - data is clean!")

# Display column names
print(f"\n=" * 70)
print(f"FEATURE COUNT: {df.shape[1]} columns")
print(f"=" * 70)
print("\nAll columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3d}. {col}")

---
## 3. Separate Features and Target Variables

We'll create two targets:
1. **G3** (final grade) - for regression
2. **risk_category** - for classification (High/Medium/Low)

In [ ]:
# Identify target columns
target_col_regression = 'G3'
target_col_classification = 'risk_category'

# Check if targets exist
if target_col_regression not in df.columns:
    raise ValueError(f"Target column '{target_col_regression}' not found in dataset!")

if target_col_classification not in df.columns:
    print(f"⚠️  '{target_col_classification}' not found. Will only prepare regression target.")
    has_classification_target = False
else:
    has_classification_target = True

print("✓ Target variables identified")

In [ ]:
# Separate features (X) and target (y)
# Drop G3 and risk_category from features
columns_to_drop = [target_col_regression]
if has_classification_target:
    columns_to_drop.append(target_col_classification)

X = df.drop(columns=columns_to_drop)
y_regression = df[target_col_regression]

if has_classification_target:
    y_classification = df[target_col_classification]

print("=" * 70)
print("FEATURES AND TARGETS SEPARATED")
print("=" * 70)
print(f"Features (X): {X.shape[0]} rows × {X.shape[1]} columns")
print(f"Target (y_regression - G3): {y_regression.shape[0]} values")
if has_classification_target:
    print(f"Target (y_classification): {y_classification.shape[0]} values")
print("\n" + "=" * 70)

In [ ]:
# Display target distribution
print("\nTarget Variable (G3) Distribution:")
print(y_regression.describe())

if has_classification_target:
    print("\nRisk Category Distribution:")
    print(y_classification.value_counts())
    print("\nPercentages:")
    print(y_classification.value_counts(normalize=True).mul(100).round(1))

---
## 4. Create Stratified Train/Validation/Test Splits

**Split Strategy:**
- 70% Training (730 samples)
- 15% Validation (157 samples)
- 15% Test (157 samples)

**Stratification:** Maintain risk category distribution across all splits.

In [ ]:
# Step 1: Split into train+val (85%) and test (15%)
stratify_col = y_classification if has_classification_target else None

if stratify_col is not None:
    print("Using stratified split based on risk_category")
else:
    print("Using random split (no stratification available)")

X_temp, X_test, y_temp_reg, y_test_reg = train_test_split(
    X, y_regression,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=stratify_col
)

if has_classification_target:
    y_temp_class = y_classification.loc[y_temp_reg.index]
    y_test_class = y_classification.loc[y_test_reg.index]

print(f"✓ Test set created: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

In [ ]:
# Step 2: Split train+val into train (70% of total) and val (15% of total)
# 70/85 ≈ 0.8235 to get 70% of original data
train_proportion = 0.70 / 0.85

stratify_temp = y_temp_class if has_classification_target else None

X_train, X_val, y_train_reg, y_val_reg = train_test_split(
    X_temp, y_temp_reg,
    train_size=train_proportion,
    random_state=RANDOM_STATE,
    stratify=stratify_temp
)

if has_classification_target:
    y_train_class = y_classification.loc[y_train_reg.index]
    y_val_class = y_classification.loc[y_val_reg.index]

print("\n=" * 70)
print("SPLITS CREATED")
print("=" * 70)
print(f"Training set:   {X_train.shape[0]:4d} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]:4d} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]:4d} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Total:          {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]:4d} samples")
print("=" * 70)

---
## 5. Verify Stratification Balance

In [ ]:
if has_classification_target:
    print("=" * 70)
    print("STRATIFICATION VERIFICATION")
    print("=" * 70)
    
    # Create comparison dataframe
    comparison = pd.DataFrame({
        'Original': y_classification.value_counts(normalize=True).mul(100).round(1),
        'Train': y_train_class.value_counts(normalize=True).mul(100).round(1),
        'Validation': y_val_class.value_counts(normalize=True).mul(100).round(1),
        'Test': y_test_class.value_counts(normalize=True).mul(100).round(1)
    })
    
    print("\nRisk Category Distribution (%):\n")
    print(comparison)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    comparison.T.plot(kind='bar', ax=ax, width=0.8)
    ax.set_xlabel('Dataset Split', fontsize=12, fontweight='bold')
    ax.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
    ax.set_title('Risk Category Distribution Across Splits', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(title='Risk Category', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Stratification is balanced across all splits!")
else:
    print("⚠️  No classification target available for stratification check")

---
## 6. Apply Feature Scaling (StandardScaler)

**CRITICAL:** Fit scaler on TRAINING data only, then transform all three sets.

This prevents **data leakage** from validation/test sets into training.

In [ ]:
# Initialize StandardScaler
scaler = StandardScaler()

print("=" * 70)
print("FEATURE SCALING")
print("=" * 70)

# Fit scaler on TRAINING data only
print("\nStep 1: Fitting scaler on training data...")
scaler.fit(X_train)
print("✓ Scaler fitted")

# Transform all three sets
print("\nStep 2: Transforming all datasets...")
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print("✓ All datasets transformed")

# Convert back to DataFrames (optional, for easier inspection)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("\n=" * 70)
print("SCALING COMPLETE")
print("=" * 70)
print(f"Training set:   {X_train_scaled.shape}")
print(f"Validation set: {X_val_scaled.shape}")
print(f"Test set:       {X_test_scaled.shape}")
print("=" * 70)

In [ ]:
# Verify scaling: check mean and std of training set
print("\nVerifying StandardScaler (Training Set):")
print(f"Mean should be ≈ 0: {X_train_scaled.mean().mean():.6f}")
print(f"Std should be ≈ 1: {X_train_scaled.std().mean():.6f}")

if abs(X_train_scaled.mean().mean()) < 0.001 and abs(X_train_scaled.std().mean() - 1) < 0.001:
    print("\n✓ Scaling verified - features are standardized!")
else:
    print("\n⚠️  Warning: Scaling may not be perfect. Check scaler.")

---
## 7. Visualize Distribution Before vs After Scaling

In [ ]:
# Select a few features to visualize
sample_features = X_train.columns[:4].tolist()  # First 4 features

fig, axes = plt.subplots(2, len(sample_features), figsize=(16, 8))

for i, feature in enumerate(sample_features):
    # Before scaling
    axes[0, i].hist(X_train[feature], bins=20, edgecolor='black', alpha=0.7)
    axes[0, i].set_title(f'Before Scaling\n{feature}', fontsize=10, fontweight='bold')
    axes[0, i].set_ylabel('Frequency' if i == 0 else '')
    
    # After scaling
    axes[1, i].hist(X_train_scaled[feature], bins=20, edgecolor='black', alpha=0.7, color='green')
    axes[1, i].set_title(f'After Scaling\n{feature}', fontsize=10, fontweight='bold')
    axes[1, i].set_ylabel('Frequency' if i == 0 else '')
    axes[1, i].axvline(0, color='red', linestyle='--', linewidth=1, label='Mean=0')

plt.suptitle('Feature Distribution: Before vs After StandardScaler', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("✓ Scaling visualization complete")

---
## 8. Check for Data Leakage

Verify that training, validation, and test sets have no overlapping samples.

In [ ]:
print("=" * 70)
print("DATA LEAKAGE CHECK")
print("=" * 70)

# Check for overlapping indices
train_indices = set(X_train.index)
val_indices = set(X_val.index)
test_indices = set(X_test.index)

# Check overlaps
train_val_overlap = train_indices.intersection(val_indices)
train_test_overlap = train_indices.intersection(test_indices)
val_test_overlap = val_indices.intersection(test_indices)

print(f"\nTrain-Val overlap: {len(train_val_overlap)} samples")
print(f"Train-Test overlap: {len(train_test_overlap)} samples")
print(f"Val-Test overlap: {len(val_test_overlap)} samples")

if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
    print("\n✓ NO DATA LEAKAGE - All splits are independent!")
else:
    print("\n⚠️  WARNING: DATA LEAKAGE DETECTED!")
    print("There are overlapping samples between splits.")

print("=" * 70)

---
## 9. Save Scaler Object

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save scaler
scaler_path = '../models/scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"✓ Scaler saved to: {scaler_path}")
print("\nThis scaler can be loaded later to transform new data:")
print("  with open('models/scaler.pkl', 'rb') as f:")
print("      scaler = pickle.load(f)")
print("      X_new_scaled = scaler.transform(X_new)")

---
## 10. Save All Splits

Save training, validation, and test sets for Week 5 modeling.

In [ ]:
# Create processed directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

print("=" * 70)
print("SAVING DATASETS")
print("=" * 70)

# Save features (X)
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
print("✓ Saved: X_train.csv")

X_val_scaled.to_csv('../data/processed/X_val.csv', index=False)
print("✓ Saved: X_val.csv")

X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
print("✓ Saved: X_test.csv")

# Save targets (y) - regression
y_train_reg.to_csv('../data/processed/y_train.csv', index=False, header=['G3'])
print("✓ Saved: y_train.csv (regression)")

y_val_reg.to_csv('../data/processed/y_val.csv', index=False, header=['G3'])
print("✓ Saved: y_val.csv (regression)")

y_test_reg.to_csv('../data/processed/y_test.csv', index=False, header=['G3'])
print("✓ Saved: y_test.csv (regression)")

# Save classification targets if available
if has_classification_target:
    y_train_class.to_csv('../data/processed/y_train_class.csv', index=False, header=['risk_category'])
    print("✓ Saved: y_train_class.csv (classification)")
    
    y_val_class.to_csv('../data/processed/y_val_class.csv', index=False, header=['risk_category'])
    print("✓ Saved: y_val_class.csv (classification)")
    
    y_test_class.to_csv('../data/processed/y_test_class.csv', index=False, header=['risk_category'])
    print("✓ Saved: y_test_class.csv (classification)")

print("\n=" * 70)
print("ALL DATASETS SAVED SUCCESSFULLY")
print("=" * 70)

---
## 11. Final Summary & Verification

In [ ]:
print("\n" + "=" * 70)
print("TASK B SUMMARY")
print("=" * 70)

print("\n✅ COMPLETED TASKS:")
print("  1. Loaded engineered features from Yugant (Task A)")
print("  2. Created stratified 70/15/15 splits")
print("  3. Applied StandardScaler (fit on train only)")
print("  4. Verified no data leakage between splits")
print("  5. Verified stratification balance")
print("  6. Saved scaler object (scaler.pkl)")
print("  7. Saved all train/val/test splits")

print("\n📊 DATASET SIZES:")
print(f"  Training:   {X_train_scaled.shape[0]:4d} samples ({X_train_scaled.shape[0]/len(X)*100:.1f}%)")
print(f"  Validation: {X_val_scaled.shape[0]:4d} samples ({X_val_scaled.shape[0]/len(X)*100:.1f}%)")
print(f"  Test:       {X_test_scaled.shape[0]:4d} samples ({X_test_scaled.shape[0]/len(X)*100:.1f}%)")
print(f"  Features:   {X_train_scaled.shape[1]} columns")

print("\n📁 SAVED FILES:")
print("  data/processed/X_train.csv")
print("  data/processed/X_val.csv")
print("  data/processed/X_test.csv")
print("  data/processed/y_train.csv")
print("  data/processed/y_val.csv")
print("  data/processed/y_test.csv")
if has_classification_target:
    print("  data/processed/y_train_class.csv")
    print("  data/processed/y_val_class.csv")
    print("  data/processed/y_test_class.csv")
print("  models/scaler.pkl")

print("\n🎯 READY FOR WEEK 5:")
print("  ✓ Data is clean, scaled, and split")
print("  ✓ No data leakage")
print("  ✓ Stratification balanced")
print("  ✓ Scaler saved for future predictions")
print("  ✓ Ready to train models!")

print("\n" + "=" * 70)
print("TASK B COMPLETE! 🚀")
print("=" * 70)

---
## Next Steps

**For Emmanuel (Task C):**
- Use these splits to build reusable data processing functions
- Create `src/data_processing.py` with functions that load these saved files

**For Hamza (Task D):**
- Validate these splits in your modeling strategy document
- Verify data quality and document any issues

**For Week 5 (Everyone):**
- Load X_train.csv and y_train.csv to train models
- Use X_val.csv and y_val.csv for hyperparameter tuning
- Reserve X_test.csv and y_test.csv for final evaluation only

---
**Author:** Pratik (Team Lead)  
**Week:** 4 (Feature Engineering & Data Preparation)  
**Status:** ✅ Complete